# 07 — Inference (New Patient)

**Inputs needed:** Raw DICOM directory for the new patient and trained checkpoints (`best_swinvit_model.pth`, `fused_cross_attention.pth`, VaRFS selection JSON).
**Outputs produced:** `results/inference_<PID>.json` with the HCC probability and an optional `attention/<PID>_attention.nii.gz` heatmap.
**Runtime:** ~3–5 minutes per patient on GPU (TotalSegmentator + radiomics dominate).


Demonstrates the design-doc *New Patient* use case using `scripts/run_inference.py`.

Given a raw DICOM directory for a new patient, this notebook:

1. Stages the DICOMs under `data/raw/<PID>/before/`.
2. Runs the full Phase 2 pipeline (phase filter → NIfTI → HU/Z-score → segment → crop).
3. Extracts radiomics, applies the saved VaRFS feature set.
4. Loads the trained SwinViT and cross-attention fusion checkpoints.
5. Outputs a single HCC risk probability + (optional) attention heatmap.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
ROOT

## Option A — call the CLI script directly

Replace `DICOM_DIR` with the path containing the new patient's DICOM files.

In [ ]:
DICOM_DIR = "/path/to/new_patient/before/"  # <-- edit me
PATIENT_ID = "INF_demo"

!python scripts/run_inference.py \
    --config configs/default.yaml \
    --dicom_dir "$DICOM_DIR" \
    --patient_id $PATIENT_ID \
    --save_heatmap

## Option B — call the underlying functions in-process

Useful when you want to inspect intermediate artifacts (NIfTI, mask, features)
without leaving the notebook.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("run_inference", ROOT / "scripts" / "run_inference.py")
run_inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_inference)

import argparse
args = argparse.Namespace(
    config=str(ROOT / "configs" / "default.yaml"),
    dicom_dir=DICOM_DIR,
    patient_id=PATIENT_ID,
    save_heatmap=True,
)
import sys as _sys
_sys.argv = [
    "run_inference.py",
    "--config", args.config,
    "--dicom_dir", args.dicom_dir,
    "--patient_id", args.patient_id,
]
if args.save_heatmap:
    _sys.argv.append("--save_heatmap")
run_inference.main()

## View the saved result

In [ ]:
import json
from src.utils.config import load_config

cfg = load_config(ROOT / "configs" / "default.yaml")
result_path = Path(cfg["paths"]["results_dir"]) / f"inference_{PATIENT_ID}.json"
if result_path.exists():
    print(json.dumps(json.loads(result_path.read_text()), indent=2))
else:
    print(f"No result yet at {result_path}")

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

heatmap = Path(cfg["paths"]["attention_dir"]) / f"{PATIENT_ID}_attention.nii.gz"
ct_path = Path(cfg["paths"]["processed_dir"]) / PATIENT_ID / "before.nii.gz"
if heatmap.exists() and ct_path.exists():
    ct = nib.load(str(ct_path)).get_fdata()
    heat = nib.load(str(heatmap)).get_fdata()
    z = int(np.unravel_index(np.argmax(heat), heat.shape)[-1])
    plt.figure(figsize=(6, 6))
    plt.imshow(ct[..., z].T, cmap="gray", origin="lower")
    plt.imshow(heat[..., z].T, cmap="jet", alpha=0.45, origin="lower")
    plt.title(f"Inference attention overlay — slice {z}")
    plt.axis("off")
    plt.show()